# 03 — Build Polymarket prices

Pull Polymarket pregame odds for every game in nba_2025_26_odds_scores.csv.

Sources:
  - Gamma API (no key): events?tag_id=745 (NBA), game events titled "Away vs. Home".
  - CLOB prices-history (no key): last price at/before tip-off, 1h interval.
  - Tip-off times: Basketball-Reference monthly schedule pages (already in bref/).

Output: polymarket_odds.csv
Columns: date, away_team, home_team, poly_home_prob, poly_away_prob,
  poly_price_ts, poly_spread_line, poly_spread_home_prob,
  poly_total_line, poly_over_prob, notes

Set `SAMPLE = True` in the driver cell to run the 5-game sample instead of the full pull. The full run is resumable: games already in the CSV are skipped.


In [ ]:
import calendar
import csv
import json
import math
import os
import re
import sys
import time
import urllib.request
import urllib.error
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

from pathlib import Path
# project root: works whether this runs from notebooks/ or the project root
def _find_proj(start):
    p = Path(start).resolve()
    for c in [p, *p.parents]:
        if (c / 'data' / 'raw' / 'nba_2025_26_odds_scores.csv').exists():
            return c
        if (c / 'data' / 'raw').is_dir() and (c / 'notebooks').is_dir():
            return c
    return p
PROJ = _find_proj(Path.cwd())
RAW = PROJ / 'data' / 'raw'
INTERIM = PROJ / 'data' / 'interim'
OUT = PROJ / 'outputs'
RES = PROJ / 'results'
CACHE = PROJ / 'cache'

ODDS_CSV = RAW / 'nba_2025_26_odds_scores.csv'
OUT_CSV = RAW / 'polymarket_odds.csv'
LOG = CACHE / 'build_polymarket.log'
BREF = CACHE / 'bref'

UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"}
GAMMA = "https://gamma-api.polymarket.com/events"
CLOB = "https://clob.polymarket.com/prices-history"
RATE = 1.0  # seconds between requests

NICK = {
    "Atlanta Hawks": "Hawks", "Boston Celtics": "Celtics",
    "Brooklyn Nets": "Nets", "Charlotte Hornets": "Hornets",
    "Chicago Bulls": "Bulls", "Cleveland Cavaliers": "Cavaliers",
    "Dallas Mavericks": "Mavericks", "Denver Nuggets": "Nuggets",
    "Detroit Pistons": "Pistons", "Golden State Warriors": "Warriors",
    "Houston Rockets": "Rockets", "Indiana Pacers": "Pacers",
    "Los Angeles Clippers": "Clippers", "Los Angeles Lakers": "Lakers",
    "Memphis Grizzlies": "Grizzlies", "Miami Heat": "Heat",
    "Milwaukee Bucks": "Bucks", "Minnesota Timberwolves": "Timberwolves",
    "New Orleans Pelicans": "Pelicans", "New York Knicks": "Knicks",
    "Oklahoma City Thunder": "Thunder", "Orlando Magic": "Magic",
    "Philadelphia 76ers": "76ers", "Phoenix Suns": "Suns",
    "Portland Trail Blazers": "Trail Blazers", "Sacramento Kings": "Kings",
    "San Antonio Spurs": "Spurs", "Toronto Raptors": "Raptors",
    "Utah Jazz": "Jazz", "Washington Wizards": "Wizards",
}

OUT_FIELDS = ["date", "away_team", "home_team", "poly_home_prob",
              "poly_away_prob", "poly_price_ts", "poly_spread_line",
              "poly_spread_home_prob", "poly_total_line", "poly_over_prob",
              "notes"]

_last_req = 0.0


def log(msg):
    line = f"{datetime.now(timezone.utc).isoformat()} {msg}"
    print(line, flush=True)
    with open(LOG, "a") as f:
        f.write(line + "\n")


def get_json(url, retries=4):
    global _last_req
    for attempt in range(retries):
        dt = time.time() - _last_req
        if dt < RATE:
            time.sleep(RATE - dt)
        _last_req = time.time()
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=30) as r:
                return json.load(r)
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503) and attempt < retries - 1:
                time.sleep(5 * (attempt + 1))
                continue
            raise
        except Exception:
            if attempt < retries - 1:
                time.sleep(5 * (attempt + 1))
                continue
            raise
    raise RuntimeError(f"failed {url}")


In [ ]:
# ---------------------------------------------------------------- tip-offs
def parse_tipoffs():
    """Parse BR monthly schedule pages -> {(date, away, home): tipoff_utc}."""
    tipoffs = {}
    for fn in sorted(os.listdir(BREF)):
        if not fn.endswith(".html"):
            continue
        html = open(BREF / fn, encoding="utf-8",
                    errors="ignore").read()
        # BR wraps the schedule table in an HTML comment; strip comment marks
        html = html.replace("<!--", "").replace("-->", "")
        # rows: date_game th, game_start_time td, visitor td, home td
        pat = re.compile(
            r'data-stat="date_game"[^>]*>(?:<a[^>]*>)?([^<]+)</a?>?</th>\s*'
            r'<td[^>]*data-stat="game_start_time"[^>]*>([^<]*)</td>\s*'
            r'<td[^>]*data-stat="visitor_team_name"[^>]*>(?:<a[^>]*>)?([^<]+)</a?>?</td>.*?'
            r'<td[^>]*data-stat="home_team_name"[^>]*>(?:<a[^>]*>)?([^<]+)</a?>?</td>',
            re.S)
        for m in pat.finditer(html):
            datestr, timestr, away, home = (x.strip() for x in m.groups())
            if not timestr:
                continue
            try:
                d = datetime.strptime(datestr, "%a, %b %d, %Y").date()
            except ValueError:
                continue
            tm = timestr.lower().replace(" ", "")
            mm = re.match(r"(\d+):(\d+)([ap])", tm)
            if not mm:
                continue
            hh, mins, ap = int(mm.group(1)), int(mm.group(2)), mm.group(3)
            if ap == "p" and hh != 12:
                hh += 12
            if ap == "a" and hh == 12:
                hh = 0
            local = datetime(d.year, d.month, d.day, hh, mins,
                             tzinfo=ZoneInfo("America/New_York"))
            tipoffs[(d.isoformat(), away, home)] = local.astimezone(timezone.utc)
    return tipoffs


In [ ]:
# ---------------------------------------------------------------- gamma
def gamma_events(day):
    """All NBA events whose endDate falls in [day 12:00Z, day+1 12:00Z]."""
    start = f"{day}T12:00:00Z"
    end = f"{(datetime.fromisoformat(day) + timedelta(days=1)).date()}T12:00:00Z"
    out, offset = [], 0
    while True:
        url = (f"{GAMMA}?tag_id=745&end_date_min={start}&end_date_max={end}"
               f"&limit=100&offset={offset}&closed=true")
        batch = get_json(url)
        if not batch:
            break
        out.extend(batch)
        if len(batch) < 100:
            break
        offset += 100
    return out


def find_event(events, away_nick, home_nick, tipoff_utc):
    """Match event by exact 'Away vs. Home' title + endDate near tip-off."""
    want = f"{away_nick} vs. {home_nick}"
    cands = []
    for e in events:
        if e.get("title") != want:
            continue
        mkts = e.get("markets", [])
        if not any(m.get("question") == want for m in mkts):
            continue
        cands.append(e)
    if not cands:
        return None, "no matching Polymarket event"
    if len(cands) == 1:
        return cands[0], ""
    # disambiguate by endDate closest to tip-off
    def dist(e):
        try:
            ed = datetime.fromisoformat(e["endDate"].replace("Z", "+00:00"))
            return abs((ed - tipoff_utc).total_seconds())
        except Exception:
            return 1e18
    cands.sort(key=dist)
    if dist(cands[0]) > 4 * 3600:
        return None, "ambiguous events, none near tip-off"
    return cands[0], "ambiguous events, picked nearest endDate"


In [ ]:
# ---------------------------------------------------------------- prices
def price_at(token, tipoff_unix):
    """Last price-history point at/before tip-off (24h window, 1h)."""
    url = (f"{CLOB}?market={token}&interval=1h&fidelity=60"
           f"&startTs={tipoff_unix - 86400}&endTs={tipoff_unix}")
    try:
        h = get_json(url)
    except urllib.error.HTTPError:
        return None, None
    hist = h.get("history", []) if isinstance(h, dict) else []
    pts = [(p["t"], p["p"]) for p in hist if p.get("t", 0) <= tipoff_unix]
    if not pts:
        return None, None
    t, p = pts[-1]
    return p, t


def outcome_token(market, outcome):
    toks = json.loads(market["clobTokenIds"])
    outs = market["outcomes"]
    if isinstance(outs, str):
        outs = json.loads(outs)
    try:
        return toks[outs.index(outcome)]
    except (ValueError, IndexError):
        return None


def parse_spread_q(q):
    """'Spread: Thunder (-7.5)' -> ('Thunder', -7.5)."""
    m = re.match(r"Spread:\s*(.+?)\s*\(([+-]?[\d.]+)\)\s*$", q)
    return (m.group(1), float(m.group(2))) if m else (None, None)


def parse_total_q(q):
    """'Rockets vs. Thunder: O/U 227.5' -> 227.5."""
    m = re.search(r"O/U\s+([\d.]+)\s*$", q)
    return float(m.group(1)) if m else None


def process_game(game, tipoff_utc, events):
    """One game -> output row dict."""
    date, away, home = game["date"], game["away_team"], game["home_team"]
    away_nick, home_nick = NICK[away], NICK[home]
    tipoff_unix = int(tipoff_utc.timestamp())
    row = {f: "" for f in OUT_FIELDS}
    row.update({"date": date, "away_team": away, "home_team": home})
    notes = []
    ev, note = find_event(events, away_nick, home_nick, tipoff_utc)
    if ev is None:
        row["notes"] = note
        return row
    if note:
        notes.append(note)
    mkts = ev.get("markets", [])

    # --- moneyline: question == title
    title = f"{away_nick} vs. {home_nick}"
    ml = next((m for m in mkts if m.get("question") == title), None)
    if ml:
        th = outcome_token(ml, home_nick)
        ta = outcome_token(ml, away_nick)
        ph, tsh = price_at(th, tipoff_unix) if th else (None, None)
        pa, tsa = price_at(ta, tipoff_unix) if ta else (None, None)
        if ph is not None:
            row["poly_home_prob"] = round(ph, 4)
        if pa is not None:
            row["poly_away_prob"] = round(pa, 4)
        ts = tsh or tsa
        if ts:
            row["poly_price_ts"] = datetime.fromtimestamp(
                ts, tz=timezone.utc).isoformat()
        if ph is None and pa is None:
            notes.append("ML market found but no price history before tip-off")
    else:
        notes.append("no moneyline market")

    # --- spread: closest line (home perspective) to game's home_spread
    target_spread = float(game["home_spread"])
    best = None  # (diff, line, market)
    for m in mkts:
        team, line = parse_spread_q(m.get("question", ""))
        if team is None:
            continue
        home_line = line if team == home_nick else -line
        diff = abs(home_line - target_spread)
        if best is None or diff < best[0] - 1e-9 or (
                abs(diff - best[0]) < 1e-9 and home_line >= target_spread > best[1]):
            best = (diff, home_line, m)
    if best:
        _, home_line, m = best
        tok = outcome_token(m, home_nick)
        p, _ = price_at(tok, tipoff_unix) if tok else (None, None)
        row["poly_spread_line"] = home_line
        if p is not None:
            row["poly_spread_home_prob"] = round(p, 4)
        else:
            notes.append("spread market found but no price history")
    else:
        notes.append("no spread market")

    # --- total: closest O/U line to game's total
    target_total = float(game["total"])
    best = None
    for m in mkts:
        line = parse_total_q(m.get("question", ""))
        if line is None:
            continue
        diff = abs(line - target_total)
        if best is None or diff < best[0] - 1e-9 or (
                abs(diff - best[0]) < 1e-9 and line >= target_total > best[1]):
            best = (diff, line, m)
    if best:
        _, line, m = best
        tok = outcome_token(m, "Over")
        p, _ = price_at(tok, tipoff_unix) if tok else (None, None)
        row["poly_total_line"] = line
        if p is not None:
            row["poly_over_prob"] = round(p, 4)
        else:
            notes.append("total market found but no price history")
    else:
        notes.append("no total market")

    row["notes"] = "; ".join(notes)
    return row


In [ ]:
# ---------------------------------------------------------------- driver
def load_games():
    with open(ODDS_CSV, newline="") as f:
        return list(csv.DictReader(f))


def load_done():
    done = set()
    if os.path.exists(OUT_CSV):
        with open(OUT_CSV, newline="") as f:
            for r in csv.DictReader(f):
                done.add((r["date"], r["away_team"], r["home_team"]))
    return done


def write_row(row):
    new = not os.path.exists(OUT_CSV)
    with open(OUT_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=OUT_FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)


def main():
    SAMPLE = False  # set True to run the 5-game sample

    games = load_games()
    log(f"loaded {len(games)} games from odds file")
    tipoffs = parse_tipoffs()
    log(f"parsed {len(tipoffs)} tip-off times from BR schedule pages")

    if SAMPLE:
        picks = [g for g in games if g["date"] == "2025-10-21"][:2]
        picks += [g for g in games if g["date"] == "2025-10-22"][:1]
        picks += [g for g in games if g["date"] == "2026-01-15"][:1]
        picks += [g for g in games if g["date"] == "2026-04-12"][:1]
        for g in picks:
            key = (g["date"], g["away_team"], g["home_team"])
            to = tipoffs.get(key)
            log(f"SAMPLE {key} tipoff={to}")
            if to is None:
                log("  no tip-off; skipping")
                continue
            evs = gamma_events(g["date"])
            row = process_game(g, to, evs)
            log("  ROW " + json.dumps(row))
        return

    done = load_done()
    log(f"resuming: {len(done)} games already done")
    by_date = {}
    for g in games:
        by_date.setdefault(g["date"], []).append(g)
    n_new = 0
    for date in sorted(by_date):
        todo = [g for g in by_date[date]
                if (g["date"], g["away_team"], g["home_team"]) not in done]
        if not todo:
            continue
        evs = gamma_events(date)
        log(f"{date}: {len(evs)} gamma events, {len(todo)} games to do")
        for g in todo:
            key = (g["date"], g["away_team"], g["home_team"])
            to = tipoffs.get(key)
            if to is None:
                row = {f: "" for f in OUT_FIELDS}
                row.update({"date": g["date"], "away_team": g["away_team"],
                            "home_team": g["home_team"],
                            "notes": "no tip-off time in BR schedule"})
                write_row(row)
                done.add(key)
                n_new += 1
                continue
            try:
                row = process_game(g, to, evs)
            except Exception as e:
                log(f"ERROR {key}: {e}")
                row = {f: "" for f in OUT_FIELDS}
                row.update({"date": g["date"], "away_team": g["away_team"],
                            "home_team": g["home_team"],
                            "notes": f"error: {e}"})
            write_row(row)
            done.add(key)
            n_new += 1
        log(f"{date}: done, total written {len(done)}")
    log(f"FINISHED: {n_new} new games, {len(done)} total in {OUT_CSV}")


In [ ]:
main()
